[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-仓库主页-181717?logo=github)](https://github.com/Luca5702/langgraph-study)

**📚 课程导航：** [01](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) · **[02](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb)** · [03](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) · [04](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb) · [05](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) · [06](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) · [07](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) · [08](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb) · [09](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) · [10](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb) · [11](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

[⬅️ 上一课：第 1 课：Hello Graph（State / Node / Edge）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) ｜ [➡️ 下一课：第 3 课：控制流（条件边 / 循环 / 并行 / Send / Command）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb)

---

# 第 2 课：State 与 Reducer —— "节点返回的更新如何合并进状态"

* 默认行为：后写覆盖前写（last-write-wins）。
* Annotated[类型, reducer]：指定合并函数 reducer(旧值, 新值) -> 合并后的值。
- operator.add：列表拼接
- add_messages：消息专用，按 id 追加/替换/删除（RemoveMessage）
* Overwrite(value)：本次更新绕过 reducer，直接替换。
* input_schema / output_schema：控制图对外的输入、输出键；
  节点内部还可以有"私有"状态。

In [1]:
#@title 🔧 第 0 步：初始化环境（在 Colab 中点击左侧 ▶ 即可自动安装依赖，本地运行自动跳过） { display-mode: "form" }
import importlib.util, subprocess, sys
if importlib.util.find_spec('langgraph') is None:
    print('⏳ 正在安装 LangGraph 环境（约 10 秒）...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph', 'langchain-core', 'langgraph-checkpoint-sqlite'])
    print('✅ 安装完成！')
else:
    import importlib.metadata as m
    print(f'✅ 环境已就绪：langgraph={m.version("langgraph")}')


✅ 环境已就绪：langgraph=1.2.22


In [2]:
import operator

from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage
from typing_extensions import Annotated, TypedDict

from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Overwrite

In [3]:
# A. 默认覆盖 vs operator.add

In [4]:
class LogState(TypedDict):
    last: str                              # 没有 reducer -> 覆盖
    history: Annotated[list[str], operator.add]  # 有 reducer -> 追加


def step_a(state: LogState):
    return {"last": "a", "history": ["a"]}


def step_b(state: LogState):
    return {"last": "b", "history": ["b"]}


def reset(state: LogState):
    # 绕过 reducer，直接清空并替换
    return {"history": Overwrite(["<reset>"])}


g = (
    StateGraph(LogState)
    .add_node(step_a).add_node(step_b).add_node(reset)
    .add_edge(START, "step_a").add_edge("step_a", "step_b").add_edge("step_b", "reset")
    .compile()
)

In [5]:
# B. 自定义 reducer

In [6]:
def keep_max(old: int, new: int) -> int:
    return max(old, new)


class MaxState(TypedDict):
    best: Annotated[int, keep_max]


g_max = (
    StateGraph(MaxState)
    .add_node("n1", lambda s: {"best": 3})
    .add_node("n2", lambda s: {"best": 1})
    .add_edge(START, "n1").add_edge("n1", "n2")
    .compile()
)

In [7]:
# C. 消息状态：add_messages / MessagesState

In [8]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]  # 等价于直接继承 MessagesState


def bot(state: ChatState):
    last = state["messages"][-1]
    return {"messages": [AIMessage(content=f"你说的是：{last.content}", id="ai-1")]}


def edit_bot(state: ChatState):
    # 同 id 的消息会被"替换"而不是追加
    return {"messages": [AIMessage(content="(已修订) 我理解了你的意思", id="ai-1")]}


def drop_first(state: ChatState):
    # RemoveMessage 按 id 删除消息
    return {"messages": [RemoveMessage(id=state["messages"][0].id)]}


g_chat = (
    StateGraph(ChatState)
    .add_node(bot).add_node(edit_bot).add_node(drop_first)
    .add_edge(START, "bot").add_edge("bot", "edit_bot").add_edge("edit_bot", "drop_first")
    .compile()
)

In [9]:
# D. 输入 / 输出 / 私有 schema

In [10]:
class InputState(TypedDict):
    question: str


class OutputState(TypedDict):
    answer: str


class OverallState(InputState, OutputState):
    pass


class PrivateState(TypedDict):
    scratch: str  # 只在节点间传递，不出现在输出中


def think(state: InputState) -> PrivateState:
    return {"scratch": f"思考『{state['question']}』..."}


def answer(state: PrivateState) -> OutputState:
    return {"answer": state["scratch"] + " 答案是 42"}


g_schema = (
    StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)
    .add_node(think).add_node(answer)
    .add_edge(START, "think").add_edge("think", "answer")
    .compile()
)

## ▶️ 运行实验

In [11]:
print("A.", g.invoke({"last": "", "history": ["init"]}))
print("B.", g_max.invoke({"best": 0}))
out = g_chat.invoke({"messages": [HumanMessage("你好", id="h-1")]})
print("C.", [(type(m).__name__, m.content) for m in out["messages"]])
print("D.", g_schema.invoke({"question": "生命的意义"}), "(注意 scratch 不在输出里)")
print("MessagesState 的字段:", MessagesState.__annotations__)

A. {'last': 'b', 'history': ['<reset>']}
B. {'best': 3}
C. [('AIMessage', '(已修订) 我理解了你的意思')]
D. {'answer': '思考『生命的意义』... 答案是 42'} (注意 scratch 不在输出里)
MessagesState 的字段: {'messages': ForwardRef('Annotated[list[AnyMessage], add_messages]', module='langgraph.graph.message')}


## ✏️ 练习：
 1. 写一个 reducer，把 dict 做深度合并（merge）。
 2. 两个并行节点同时写同一个"没有 reducer"的键，会报什么错？（提示：InvalidUpdateError，第 3 课验证）

In [12]:
# 在这里写你的练习代码 👇


In [13]:
#@title 💡 展开查看参考答案与解析（可点击 ▶ 直接运行验证） { display-mode: "form" }
# 练习 1：自定义字典深度合并 reducer
def deep_merge(old: dict | None, new: dict) -> dict:
    base = dict(old or {})
    for k, v in new.items():
        if isinstance(v, dict) and isinstance(base.get(k), dict):
            base[k] = deep_merge(base[k], v)
        else:
            base[k] = v
    return base

class DictState(TypedDict):
    profile: Annotated[dict, deep_merge]

g_merge = (
    StateGraph(DictState)
    .add_node("n1", lambda s: {"profile": {"user": {"name": "小明"}, "score": 10}})
    .add_node("n2", lambda s: {"profile": {"user": {"city": "上海"}, "score": 20}})
    .add_edge(START, "n1").add_edge("n1", "n2")
    .compile()
)
print("练习 1 深度合并结果:", g_merge.invoke({"profile": {}}))

# 练习 2：两个并行节点同时写同一个「没有 reducer」的键 -> 抛出 InvalidUpdateError
from langgraph.errors import InvalidUpdateError
class NoReducerState(TypedDict):
    val: str

g_conflict = (
    StateGraph(NoReducerState)
    .add_node("p1", lambda s: {"val": "from_p1"})
    .add_node("p2", lambda s: {"val": "from_p2"})
    .add_edge(START, "p1").add_edge(START, "p2")
    .compile()
)
try:
    g_conflict.invoke({"val": "init"})
except InvalidUpdateError as e:
    print("练习 2 捕获到预期异常:", type(e).__name__, "->", str(e).splitlines()[0])

练习 1 深度合并结果: {'profile': {'user': {'name': '小明', 'city': '上海'}, 'score': 20}}
练习 2 捕获到预期异常: InvalidUpdateError -> At key 'val': Can receive only one value per step. Use an Annotated key to handle multiple values.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-仓库主页-181717?logo=github)](https://github.com/Luca5702/langgraph-study)

**📚 课程导航：** [01](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) · **[02](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb)** · [03](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) · [04](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb) · [05](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) · [06](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) · [07](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) · [08](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb) · [09](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) · [10](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb) · [11](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

[⬅️ 上一课：第 1 课：Hello Graph（State / Node / Edge）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) ｜ [➡️ 下一课：第 3 课：控制流（条件边 / 循环 / 并行 / Send / Command）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb)

---